# Met Museum API Exploration

This notebook explores the Met Museum Collection API as part of building a data pipeline that extracts artwork metadata and prepares it for analysis in PostgreSQL.

**Goals**
- Understand the API structure
- Fetch all object IDs
- Inspect the response format
- Fetch details for a small sample of objects
- Identify useful fields for later transformation and loading into PostgreSQL

In [2]:
import requests

In [3]:
# Base endpoint for retrieving all Met object IDs
objects_url = "https://collectionapi.metmuseum.org/public/collection/v1/objects"

In [4]:
# Function to fetch all object IDs from the API
def fetch_object_ids():
    response = requests.get(objects_url)
    objects_json = response.json()
    
    return objects_json

In [5]:
# Fetch object IDs and inspect the structure of the API response
objects = fetch_object_ids()

print("Keys:", objects.keys())
print("Total objects:", objects["total"])
print("First 5 IDs:", objects["objectIDs"][:5])

Keys: dict_keys(['total', 'objectIDs'])
Total objects: 500382
First 5 IDs: [1, 2, 3, 4, 5]


# Notes on the `/objects` endpoint

The `/objects` endpoint does not return artwork details directly.

It returns:
- `total`: total number of objects in the collection
- `objectIDs`: a list of object IDs

These IDs are then used to query the object details endpoint for individual artworks.

In [6]:
# Function to fetch details for the first 5 objects
def fetch_object_details(objects):
    all_objects = []
    
    for object_id in objects["objectIDs"][:5]:
        object_url = f"https://collectionapi.metmuseum.org/public/collection/v1/objects/{object_id}"
        response = requests.get(object_url)
        object_json = response.json()
        
        all_objects.append(object_json)

    return all_objects

In [7]:
details = fetch_object_details(objects)

In [8]:
# Confirm that 5 object records were returned
len(details)

5

In [9]:
# View the available fields for one object
details[0].keys()

dict_keys(['objectID', 'isHighlight', 'accessionNumber', 'accessionYear', 'isPublicDomain', 'primaryImage', 'primaryImageSmall', 'additionalImages', 'constituents', 'department', 'objectName', 'title', 'culture', 'period', 'dynasty', 'reign', 'portfolio', 'artistRole', 'artistPrefix', 'artistDisplayName', 'artistDisplayBio', 'artistSuffix', 'artistAlphaSort', 'artistNationality', 'artistBeginDate', 'artistEndDate', 'artistGender', 'artistWikidata_URL', 'artistULAN_URL', 'objectDate', 'objectBeginDate', 'objectEndDate', 'medium', 'dimensions', 'measurements', 'creditLine', 'geographyType', 'city', 'state', 'county', 'country', 'region', 'subregion', 'locale', 'locus', 'excavation', 'river', 'classification', 'rightsAndReproduction', 'linkResource', 'metadataDate', 'repository', 'objectURL', 'tags', 'objectWikidata_URL', 'isTimelineWork', 'GalleryNumber'])

In [10]:
# Inspect a few useful fields
details[0]["title"], details[0]["artistDisplayName"], details[0]["objectDate"]

('One-dollar Liberty Head Coin', 'James Barton Longacre', '1853')

## Inspecting object records

The object details endpoint returns a large JSON record containing many attributes describing each artwork.

Initial observations include:

- `objectID` appears to be the unique identifier for each artwork.
- Fields such as `title`, `artistDisplayName`, and `objectDate` provide descriptive metadata.
- Some fields, such as `accessionNumber`, are descriptive but not necessarily unique.
- Several image-related fields are present and may be excluded later if they are not required for analysis.

Because each object contains many attributes, the transformation step will later select only the fields needed for analysis.

## Testing larger extraction batches

After confirming that the API calls worked for small samples, the next step was to test a larger batch of objects.

During early testing on batches of around 100 object IDs, the extraction process appeared to stop around IDs in the 80–90 range.

However, repeated runs showed that the failure point was not consistent. This suggested the issue was not tied to a specific object record.

The behaviour pointed instead to intermittent API responses or network instability, which motivated improvements to the extraction logic.

## Improving the extraction logic

To make the extraction step more robust, the function was refactored to:

- parameterise the number of objects processed using a `limit` argument
- validate API responses using HTTP status codes
- handle request exceptions
- record failed object IDs without stopping the extraction

This ensures the pipeline can continue processing even when some requests fail.

In [11]:
from tqdm import tqdm

# Improved extraction function
def fetch_object_details(objects, limit=100):
    all_objects = []
    failed_ids = {}

    for object_id in tqdm(objects["objectIDs"][:limit]):
        try:
            object_url = f"https://collectionapi.metmuseum.org/public/collection/v1/objects/{object_id}"
            response = requests.get(object_url, timeout=10)

            if response.status_code == 200:
                object_json = response.json()
                all_objects.append(object_json)
            else:
                failed_ids[object_id] = response.status_code

        except Exception:
            failed_ids[object_id] = "exception"

    return all_objects, failed_ids

In [12]:
# Run extraction on a larger test batch
details, failed_ids = fetch_object_details(objects, limit=100)

print("Successful records:", len(details))
print("Failed records:", len(failed_ids))

100%|██████████| 100/100 [00:37<00:00,  2.68it/s]

Successful records: 74
Failed records: 26


In [13]:
# Preview a few failed requests
list(failed_ids.items())[:5]

[(81, 403), (84, 403), (85, 403), (86, 403), (87, 403)]

Failed requests in testing returned HTTP 403 responses.

According to the Met API documentation, no API key is required, so these failures may reflect intermittent API restrictions or inaccessible object records rather than missing authentication.

In [14]:
# Confirm that extracted records contain usable fields
details[0]["title"]

'One-dollar Liberty Head Coin'

## Extraction result

Testing the improved extraction function on the first 100 object IDs produced both successful and failed responses.

Because failed requests are recorded rather than stopping execution, the pipeline can continue processing remaining records even when some API calls do not return valid responses.

The extraction logic developed in this notebook was later moved into a dedicated `extract.py` module, while `main.py` orchestrates the pipeline execution.

### API Rate Behaviour

When testing larger batches of object detail requests, many responses returned HTTP 403 status codes.

This suggests the API may enforce practical request limits despite documentation indicating no authentication is required.

The extraction function therefore records failed IDs and continues processing to ensure the pipeline remains robust.

## API Behaviour During Batch Extraction

During testing of larger batches of object detail requests, a significant number of responses returned HTTP 403 status codes.

Additional tests using different slices of the object ID list showed that success rates varied depending on the range of IDs being queried. Some slices returned mostly successful responses, while others resulted in many 403 errors.

A brief delay between requests (`time.sleep`) was tested to determine whether the failures were caused by request rate limits. This did not consistently improve success rates.

### Experimental Results

| Slice | Limit | Sleep | Success | Failed | Failure Code |
|------|------|------|------|------|------|
| 0–500 | 500 | 0.05 | 79 | 421 | 403 |
| 10000–10500 | 500 | 0.05 | 275 | 225 | 403 |
| 300000–300500 | 500 | 0.05 | 290 | 210 | 403 |
| 300000–300500 | 500 | 0.00 | 95 | 405 | 403 |
| 300000–300500 | 500 | 0.10 | 80 | 420 | 403 |
| 300000–300100 | 100 | 0.00 | 100 | 0 | N/A |
| 300100–300200 | 100 | 0.00 | 99 | 1 | 403 |
| 300200–300300 | 100 | 0.00 | 82 | 18 | 403 |
| 300300–300400 | 100 | 0.00 | 96 | 4 | 403 |
| 300400–300500 | 100 | 0.00 | 100 | 0 | N/A |

These tests showed that large uninterrupted request loops (e.g. 500 requests in a single run) often produced a high number of HTTP 403 responses. For example, a 500-ID batch returned fewer than 100 successful responses, while splitting the same range into 100-ID batches resulted in success rates above 80–100%.

This behaviour suggests that the API may limit sustained request streams rather than strictly enforcing a per-request rate limit. To ensure the extraction pipeline remains robust, the implementation records failed IDs and continues processing successful responses.